# Lab 11: Government Data APIs

*INFO 4617/5617 · Week 11 · Chapter 11, [Government Data APIs](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html)*

This week every class is live coding: one lab in three parts, one a day. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in. After each step, a **💬 Discuss** box holds questions to talk over before the next one; click it to open it.

We'll use three public APIs, and each one shows an idea that holds for APIs in general:
- **Part 1, Monday:** the **Census Bureau**, and *what can I ask for?*, starting from a table on its website;
- **Part 2, Wednesday:** more of the Census, then **FRED**, the Federal Reserve Bank of St. Louis's data service, and *let the server compute*;
- **Part 3, Friday:** more of FRED, then the **Energy Information Administration (EIA)**, and *let the server filter*.

The chapter's fourth API, Congress.gov's, and its join of two agencies' data are there to read on your own.

Along the way, short ⏱ microsprints ask you to change one thing, alone or in pairs, and share what you find.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. Request three free keys and store them as environment variables **before you start Jupyter**. The chapter's ["Setting Environment Variables"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#setting-environment-variables) shows how; the simplest for this course stores them with the `webdata` environment, one line per key in a terminal (on Windows, the Anaconda Prompt), then a fresh start of Jupyter:
- `CENSUS_KEY`, from the [Census Bureau](https://api.census.gov/data/key_signup.html), for Monday and Wednesday; click the activation link in its email before you use it;
- `FRED_KEY`, from [FRED](https://fred.stlouisfed.org/docs/api/api_key.html), for Wednesday and Friday;
- `EIA_KEY`, from the [EIA](https://www.eia.gov/opendata/register.php), for Friday.

Each part reads only the keys it needs, and its setup step says what to do if one is missing.

On Wednesday and Friday, start at that day's setup step, or run every cell above where we stopped (**Run → Run All Above Selected Cell**).

> **Instructor key.** KEYNOTE

# Part 1 — Monday: the Census, and what can I ask for?

Every number on the Census Bureau's website also comes from its API. We start from one table on the website, find the same table in the API's catalog, read what it holds, and ask the API for the website's number. Finding out what an API offers, from the API itself, is called *discovery*.

## Step 0 — Set up, and check your key (8 minutes)

An API key identifies you, counts your requests, and is yours alone. A key typed into a notebook travels with the notebook: into a GitHub repository, a class submission, a classmate's laptop. So read it from an environment variable, as the chapter's ["Reading the Keys in Python"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#reading-the-keys-in-python) does, and check first that Python can see it, without printing it. Define the User-Agent too: these APIs ask for a key, but every request still says who sent it.

If it says `MISSING`, Python can't see the key, and `os.environ[...]` will stop with a `KeyError`. Jupyter sees only the environment variables that were set **before it started**, and only in **the place it was started from**:

- You ran `export CENSUS_KEY=...` in one terminal, but Jupyter was already running, or was started from another terminal, or from Anaconda Navigator (which, on a Mac, doesn't read `~/.zshrc`).
- On Windows, `setx` reaches only prompts opened after it.

The dependable fix is the chapter's first way in ["Setting Environment Variables"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#setting-environment-variables): store the key with the `webdata` environment, `conda env config vars set CENSUS_KEY="your-census-key" -n webdata`, then quit Jupyter and start it again. If you can't restart now, run the last cell of this step instead: [`getpass()`](https://docs.python.org/3/library/getpass.html) asks for the key in a box that hides what you paste, so it never appears in the notebook, and Python forgets it when the kernel restarts.

In [ ]:
# Import os, requests, pandas (as pd), and matplotlib.pyplot (as plt)
# Define HEADERS, whose "User-Agent" is "WebDataScience/1.0 (your-email@colorado.edu)", with your email
# Print whether "CENSUS_KEY" is in os.environ: "set" or "MISSING", never the key itself
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
print("CENSUS_KEY:", "set" if "CENSUS_KEY" in os.environ else "MISSING")

In [ ]:
# If it said set: read CENSUS_KEY into census_key with os.environ[...]
census_key = os.environ["CENSUS_KEY"]

In [ ]:
# Only if it said MISSING and you can't restart Jupyter now: ask for the key with getpass, which hides what you paste
from getpass import getpass
census_key = getpass("Paste your Census key: ")

<details>
<summary>💬 Discuss</summary>

- In week 10 we told Wikipedia who we are with a User-Agent. The Census Bureau also asks for a key: what can a key do that a User-Agent can't?
- Where could a key typed into this notebook end up: a commit, a screenshot, a shared notebook? The Census Bureau takes the key in the URL, so `response.url` contains it too. Which of those could you take back?
- Why does a terminal's `export` reach a Jupyter started from that terminal, but not one started from Anaconda Navigator?

</details>

## Step 1 — From the website to the catalog: groups (10 minutes)

On [data.census.gov](https://data.census.gov/), the Census Bureau's website for its tables, table **B19013** gives Boulder County's median household income: \$103,994, ±\$2,268, in the American Community Survey's (ACS) five-year estimates for 2020–2024 (the chapter's tour, [figure 11.1](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#fig-census-table)). In the API, that table is a **group**, and the dataset lists all of its groups in a catalog of its own, `groups.json` (or, for people, [groups.html](https://api.census.gov/data/2024/acs/acs5/groups.html)). Load it into a DataFrame and look at a sample: each group has a `name`, the table's ID, and a `description`, its title. Then search the descriptions for the website's words, as the chapter's ["What Can I Ask For? Variables"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#what-can-i-ask-for-variables) does, with [`str.contains()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html).

In [ ]:
# groups: a DataFrame of the JSON's "groups" from "https://api.census.gov/data/2024/acs/acs5/groups.json", its name and description
# Print how many groups there are, and show the first 10, sorted by name
groups_url = "https://api.census.gov/data/2024/acs/acs5/groups.json"
groups = pd.DataFrame(requests.get(groups_url, headers=HEADERS).json()["groups"])[["name", "description"]]
print(len(groups), "groups")
groups.sort_values("name").head(10)

In [ ]:
# Show the groups whose description contains "median household income" (case=False), sorted by name
groups[groups["description"].str.contains("median household income", case=False)].sort_values("name")

### ⏱ Microsprint (4 minutes, alone or in pairs)

Find a table for a question of your own.

**Do:** search the descriptions for another topic, such as "median gross rent", "travel time to work", or "internet". The key searches for "median gross rent".
**Share:** your words, how many groups matched, and the one you'd use.

In [ ]:
# Microsprint: show the groups whose description contains your words (case=False), sorted by name
groups[groups["description"].str.contains("median gross rent", case=False)].sort_values("name")

<details>
<summary>💬 Discuss</summary>

- `B19013` came back with nine siblings, `B19013A` to `B19013I`. What do the letters add, and when would you want one of them instead?
- The website calls B19013 a table; the API calls it a group. What else did you have to translate from the website's words to the API's?
- Your search found several tables. How did you choose one?

</details>

## Step 2 — A group's variables (8 minutes)

A group holds **variables**, the columns of the website's table, and each group has its own catalog: `groups/B19013.json` (or, for people, [groups/B19013.html](https://api.census.gov/data/2024/acs/acs5/groups/B19013.html)). A variable's name is the group, a line number, and a letter: `B19013_001E` is line 1's **E**stimate, and `B19013_001M` its **M**argin of error, the "±" on the website. Names ending in `A` are annotations, which say when an estimate is missing or capped. Print each variable with its label, sorted by name.

In [ ]:
# variables: the JSON's "variables" from "https://api.census.gov/data/2024/acs/acs5/groups/B19013.json"
# Loop over sorted(variables.items()), and print each name and its "label"
variables = requests.get("https://api.census.gov/data/2024/acs/acs5/groups/B19013.json", headers=HEADERS).json()["variables"]
for name, about in sorted(variables.items()):
    print(name, "|", about["label"])

### ⏱ Microsprint (4 minutes, alone or in pairs)

A table with more lines.

**Do:** print the estimates (names ending in `E`) of group `B25034`, Year Structure Built, which we use on Wednesday. Each line is a decade.
**Share:** how many lines it has, and what line `002` counts.

In [ ]:
# Microsprint: request groups/B25034.json; print each variable whose name ends in "E", with its label, sorted by name
housing_catalog = requests.get("https://api.census.gov/data/2024/acs/acs5/groups/B25034.json", headers=HEADERS).json()["variables"]
for name, about in sorted(housing_catalog.items()):
    if name.endswith("E"):
        print(name, "|", about["label"])

<details>
<summary>💬 Discuss</summary>

- The website shows the estimate and the margin side by side. In the API they are two variables. Why might an API split them?
- What do the `!!` in a label separate, and how would you get just the last part?
- Which of these variables would you ask for to rebuild the website's table exactly?

</details>

## Step 3 — The website's number, from the API (10 minutes)

Now ask the API for what the website showed. A request names the variables with `get`, the place with `for`, and what that place sits inside with `in`, as the chapter's ["Querying for a Place"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#querying-for-a-place) does. Places are named by [FIPS codes](https://www.census.gov/library/reference/code-lists/ansi.html), strings of digits: Colorado is state `08`, Boulder County is county `013` within it, and the city of Boulder is place `07850`. The answer is a list of lists whose first row is the header, and every value is a string, so make a DataFrame and convert the numbers with [`pd.to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html). It should match the website: \$103,994, ±\$2,268. Then ask for the city, first without `in`, and read what the API says.

In [ ]:
# census_base = "https://api.census.gov/data/2024/acs/acs5"
# params: get="NAME,B19013_001E,B19013_001M", for="county:013", in="state:08", key=census_key
# Request it; county: a DataFrame from rows 1 onward, with row 0 as the columns; convert the two numbers; show it
census_base = "https://api.census.gov/data/2024/acs/acs5"
params = {"get": "NAME,B19013_001E,B19013_001M", "for": "county:013", "in": "state:08", "key": census_key}
data = requests.get(census_base, params=params, headers=HEADERS).json()
county = pd.DataFrame(data[1:], columns=data[0])
county[["B19013_001E", "B19013_001M"]] = county[["B19013_001E", "B19013_001M"]].apply(pd.to_numeric)
county

In [ ]:
# The city of Boulder is place 07850. Ask for it without "in": response = requests.get(...) with get, for="place:07850", and key
# Print response.status_code and response.text: what does the API say a place needs?
response = requests.get(census_base, params={"get": "NAME,B19013_001E", "for": "place:07850", "key": census_key}, headers=HEADERS)
print(response.status_code, response.text)

In [ ]:
# Now with in="state:08": city, a DataFrame as above, with the two numbers converted; show it
params = {"get": "NAME,B19013_001E,B19013_001M", "for": "place:07850", "in": "state:08", "key": census_key}
data = requests.get(census_base, params=params, headers=HEADERS).json()
city = pd.DataFrame(data[1:], columns=data[0])
city[["B19013_001E", "B19013_001M"]] = city[["B19013_001E", "B19013_001M"]].apply(pd.to_numeric)
city

### ⏱ Microsprint (3 minutes, alone or in pairs)

Another county.

**Do:** ask for the same two variables for another Colorado county; find its code in the [FIPS code lists](https://www.census.gov/library/reference/code-lists/ansi.html) under "Counties". The key uses Larimer County, `069`. Then find the same county in table B19013 on data.census.gov.
**Share:** the county, its median household income and margin, and whether the website agrees.

In [ ]:
# Microsprint: set params["for"] to your county (the key: "county:069"), request, and show the DataFrame
params["for"] = "county:069"
data = requests.get(census_base, params=params, headers=HEADERS).json()
pd.DataFrame(data[1:], columns=data[0])

<details>
<summary>💬 Discuss</summary>

- Did the API's number match the website's? If it hadn't, what would you check first: the variable, the place, or the year?
- Boulder County's median is \$103,994 and the city's \$87,493. How can a city's median be lower than its county's?
- Why must `08` stay a string? What would `int("08")` lose?

</details>

## End of Part 1

On Wednesday, Part 2 compares places, checks a number against its margin, and finds the places with no estimate; then it asks FRED to compute for us. It starts with its own setup step, which reads the keys again.

# Part 2 — Wednesday: more of the Census; then FRED, and let the server compute

We finish what Monday started: comparing places, reading a margin of error, and the places the survey couldn't measure. Then FRED (Federal Reserve Economic Data), from the Federal Reserve Bank of St. Louis, transforms a series for us before it sends it.

## Step 4 — Set up again (5 minutes)

Part 2 starts fresh, so a student who missed Monday can start here. Import what we need, define the User-Agent, and check two keys before reading them: `CENSUS_KEY` and `FRED_KEY`. [`os.environ["NAME"]`](https://docs.python.org/3/library/os.html#os.environ) stops with a `KeyError` that names a key Python can't see; Step 0 says why that happens and how to fix it, and the last cell here uses [`getpass()`](https://docs.python.org/3/library/getpass.html) if you can't restart Jupyter now.

In [ ]:
# Import os, requests, pandas (as pd), and matplotlib.pyplot (as plt); define HEADERS as in Step 0, with your email
# For "CENSUS_KEY" and "FRED_KEY", print whether each is set or MISSING
# census_base = "https://api.census.gov/data/2024/acs/acs5"
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
for name in ["CENSUS_KEY", "FRED_KEY"]:
    print(name + ":", "set" if name in os.environ else "MISSING")
census_base = "https://api.census.gov/data/2024/acs/acs5"

In [ ]:
# If both are set: read them into census_key and fred_key with os.environ[...]
census_key = os.environ["CENSUS_KEY"]
fred_key = os.environ["FRED_KEY"]

In [ ]:
# Only for a key that's MISSING, if you can't restart Jupyter now: ask for it with getpass
from getpass import getpass
census_key = getpass("Paste your Census key: ")
fred_key = getpass("Paste your FRED key: ")

<details>
<summary>💬 Discuss</summary>

- Monday's key went in a URL parameter called `key`. FRED's is `api_key`. Why might every API name the same thing differently?
- Did your key work today? If it didn't on Monday, what changed?

</details>

## Step 5 — Compare two places (13 minutes)

Several places can go in one request: `place:07850,45970` is Boulder and Longmont, as in the chapter's ["Comparing Places"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#comparing-places). Ask for all ten decades of group `B25034`, lines `002` (built 2020 or later) to `011` (1939 or earlier); a [list comprehension](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions) with an [f-string](https://docs.python.org/3/tutorial/inputoutput.html#formatted-string-literals) writes the ten codes. Then draw the chapter's [figure 11.5](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#fig-housing-boulder-longmont): one pair of bars per decade, labeled with the words from the group's catalog, `groups/B25034.json` (Step 2's microsprint), and turned upside down first, so that time runs left to right, from the oldest decade to the newest. [`DataFrame.plot.bar()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.bar.html) draws a pair for each row.

In [ ]:
# housing_vars: "B25034_002E" to "B25034_011E", as [f"B25034_{line:03d}E" for line in range(2, 12)]
# params: get="NAME," plus housing_vars joined with commas, for="place:07850,45970", in="state:08", key=census_key; request it
# df_compare: a DataFrame from rows 1 onward, with row 0 as the columns, housing_vars converted; show it
housing_vars = [f"B25034_{line:03d}E" for line in range(2, 12)]
params = {"get": "NAME," + ",".join(housing_vars), "for": "place:07850,45970", "in": "state:08", "key": census_key}
data = requests.get(census_base, params=params, headers=HEADERS).json()
df_compare = pd.DataFrame(data[1:], columns=data[0])
df_compare[housing_vars] = df_compare[housing_vars].apply(pd.to_numeric)
df_compare

In [ ]:
# housing_catalog: the "variables" of groups/B25034.json; labels: each housing variable's label, the part after its last "!!"
# housing: df_compare indexed by NAME, its housing_vars renamed with labels, and transposed
# Plot housing.iloc[::-1] (oldest decade first) as bars; a title, a y label, no x label, a legend with no title
housing_catalog = requests.get("https://api.census.gov/data/2024/acs/acs5/groups/B25034.json", headers=HEADERS).json()["variables"]
labels = {var: housing_catalog[var]["label"].split("!!")[-1] for var in housing_vars}
housing = df_compare.set_index("NAME")[housing_vars].rename(columns=labels).T

ax = housing.iloc[::-1].plot.bar(figsize=(8, 4.5))
ax.set(title="Housing units by year built: Boulder and Longmont, ACS 2020-2024", ylabel="Housing units", xlabel="")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.legend(title="")
plt.show()

### ⏱ Microsprint (6 minutes, alone or in pairs)

A short version of Additional Exercise 1. Cities differ in size, so raw counts don't compare fairly; shares do.

**Do:** ask for the total (`B25034_001E`) and the three decades since 2000 (`_002E` to `_004E`) for five Colorado places, and compute the share of each city's housing built in 2000 or later. The key uses Boulder, Longmont, Fort Collins (`27425`), Lakewood (`43000`), and Castle Rock (`12415`); look up others in the [FIPS code lists](https://www.census.gov/library/reference/code-lists/ansi.html).
**Share:** the city with the largest share, the one with the smallest, and the two shares.

In [ ]:
# Microsprint: set params["get"] to NAME and B25034_001E to B25034_004E, params["for"] to five places, and request
# df_five: indexed by NAME, the four counts converted with pd.to_numeric
# since_2000: 002E + 003E + 004E; share: since_2000 / 001E; show share, largest first
params["get"] = "NAME,B25034_001E,B25034_002E,B25034_003E,B25034_004E"
params["for"] = "place:07850,45970,27425,43000,12415"
data = requests.get(census_base, params=params, headers=HEADERS).json()
df_five = pd.DataFrame(data[1:], columns=data[0]).set_index("NAME")
df_five = df_five[["B25034_001E", "B25034_002E", "B25034_003E", "B25034_004E"]].apply(pd.to_numeric)
since_2000 = df_five["B25034_002E"] + df_five["B25034_003E"] + df_five["B25034_004E"]
share = since_2000 / df_five["B25034_001E"]
share.sort_values(ascending=False).round(3)

<details>
<summary>💬 Discuss</summary>

- Which of your five cities built the most of its housing since 2000, and which the least? What might explain the difference?
- Boulder's biggest decade was the 1970s, and Longmont has more units than Boulder from every decade since the 1990s. The chapter names one likely reason, Boulder's 1976 cap on building permits, the Danish Plan. What would you need to know to say it was the reason?
- The chart counts the units standing in 2020–2024, by the year they were built. What does it leave out, and what does a five-year estimate mean for a count labeled "Built 2020 or later"?

</details>

## Step 6 — Check a number against its margin (6 minutes)

A survey's estimate comes with a [margin of error](https://www.census.gov/content/dam/Census/library/publications/2020/acs/acs_general_handbook_2020_ch07.pdf), the variable with the same name ending in `M`; the ACS gives it at 90% confidence. Two estimates whose ranges overlap can't be told apart. Ask for the median household income and its margin for Boulder and Longmont, and compute each city's range: the estimate minus and plus its margin. The chapter's ["Querying for a Place"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#querying-for-a-place) reads the same margin on the website as "±".

In [ ]:
# params: get="NAME,B19013_001E,B19013_001M", for="place:07850,45970", in="state:08", key=census_key; request it
# incomes: indexed by NAME, the two columns converted; add low (estimate - margin) and high (estimate + margin); show it
params = {"get": "NAME,B19013_001E,B19013_001M", "for": "place:07850,45970", "in": "state:08", "key": census_key}
data = requests.get(census_base, params=params, headers=HEADERS).json()
incomes = pd.DataFrame(data[1:], columns=data[0]).set_index("NAME")
incomes = incomes[["B19013_001E", "B19013_001M"]].apply(pd.to_numeric)
incomes["low"] = incomes["B19013_001E"] - incomes["B19013_001M"]
incomes["high"] = incomes["B19013_001E"] + incomes["B19013_001M"]
incomes

<details>
<summary>💬 Discuss</summary>

- Do the two ranges overlap? What can you say, and not say, about which city's households earn more?
- Boulder's margin is twice Longmont's. What about the city could make its estimate less certain?
- Which checks like this one could a pipeline run on its own, every time it runs?

</details>

## Step 7 — When there's no estimate (9 minutes)

A wildcard asks for every place of a kind: `for=place:*` with `in=state:08` returns all of Colorado's cities, towns, and census-designated places, as in the chapter's ["Every Place in a State, and a Number That Isn't"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#every-place-in-a-state-and-a-number-that-isnt). Ask for each one's median household income, and [`describe()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html) the column before you trust it: look at its minimum and its maximum.

Eighty-three of the 482 places have no estimate in the 2020–2024 survey (72 did in 2018–2022): too few households answered for the Census Bureau to compute one. The API doesn't leave the value out. It sends a number that can't be an income, `-666666666`, and `-222222222` for its margin of error; the Bureau's [notes on estimate and annotation values](https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html) list what each code means. Then look closely at four places: ask for the population (`B01003_001E`), the median income, and its margin for Boulder and three small towns, Bonanza (`07571`), Campo (`11645`), and Carbonate (`12030`), and set every negative value aside with [`mask()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mask.html), which turns it into a missing value, `NaN`.

In [ ]:
# params: get="NAME,B19013_001E", for="place:*", in="state:08", key=census_key; request it
# places: a DataFrame of the rows; places["income"]: B19013_001E converted with pd.to_numeric
# Print how many places came back, and describe() the income column
params = {"get": "NAME,B19013_001E", "for": "place:*", "in": "state:08", "key": census_key}
data = requests.get(census_base, params=params, headers=HEADERS).json()
places = pd.DataFrame(data[1:], columns=data[0])
places["income"] = pd.to_numeric(places["B19013_001E"])
print(len(places), "places")
places["income"].describe().round()

In [ ]:
# params: get="NAME,B01003_001E,B19013_001E,B19013_001M", for="place:07850,07571,11645,12030", in="state:08", key=census_key
# Request it; towns: indexed by NAME, the three columns converted with pd.to_numeric; show towns
params = {"get": "NAME,B01003_001E,B19013_001E,B19013_001M",
          "for": "place:07850,07571,11645,12030", "in": "state:08", "key": census_key}
data = requests.get(census_base, params=params, headers=HEADERS).json()
towns = pd.DataFrame(data[1:], columns=data[0]).set_index("NAME")
towns = towns[["B01003_001E", "B19013_001E", "B19013_001M"]].apply(pd.to_numeric)
towns

In [ ]:
# Print the mean of the median-income column as it is; then mask every value below zero, and print the mean again
print(f"Mean with the codes in: {towns['B19013_001E'].mean():,.0f}")
towns = towns.mask(towns < 0)
print(f"Mean with them masked: {towns['B19013_001E'].mean():,.0f}")
towns

<details>
<summary>💬 Discuss</summary>

- `describe()` showed both ends of the scale: `-666666666`, and a largest median of 250,001. One is a code for no estimate, and one is a cap: "\$250,000 or more". What would each do to a ranking of places by income, and would you have noticed?
- Why might an API send a code like this rather than leave the value empty? What does a missing value lose that the code keeps?
- Campo has an estimate this time, with a margin of error almost as large. Is that more useful than Bonanza's code, or less?
- Carbonate has no residents, yet it has a row. What does the Census count as a place, and what does that mean for a table of "every place"?

</details>

## Step 8 — A FRED series, and what it is (6 minutes)

FRED (Federal Reserve Economic Data) names each of its 800,000 series with an ID, which you find on the series' web page (the chapter's tour, [figure 11.2](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#fig-fred-series)). Before downloading a series, ask the API's [`series` endpoint](https://fred.stlouisfed.org/docs/api/fred/series.html) what it is. Then ask the [`series/observations` endpoint](https://fred.stlouisfed.org/docs/api/fred/series_observations.html) for its values. FRED sends every value as a string and writes a missing one as `"."`, so convert them with [`pd.to_numeric(errors="coerce")`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html), which turns `"."` into `NaN`. The series is Boulder County's median household income, as in the chapter's ["Finding Series IDs"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#finding-series-ids).

In [ ]:
# fred_base = "https://api.stlouisfed.org/fred"; params: series_id="MHICO08013A052NCEN", api_key=fred_key, file_type="json"
# Request f"{fred_base}/series" with params and HEADERS; info: the JSON's "seriess"[0]; print its title, frequency, and units
fred_base = "https://api.stlouisfed.org/fred"
params = {"series_id": "MHICO08013A052NCEN", "api_key": fred_key, "file_type": "json"}
info = requests.get(f"{fred_base}/series", params=params, headers=HEADERS).json()["seriess"][0]
print(info["title"], "|", info["frequency"], "|", info["units"])

In [ ]:
# Add observation_end="2024-12-31" to params; request f"{fred_base}/series/observations"
# income: a DataFrame of the JSON's "observations", date and value only; convert both; show the last 3 rows
params["observation_end"] = "2024-12-31"
data = requests.get(f"{fred_base}/series/observations", params=params, headers=HEADERS).json()
income = pd.DataFrame(data["observations"])[["date", "value"]]
income["date"] = pd.to_datetime(income["date"])
income["value"] = pd.to_numeric(income["value"], errors="coerce")
income.tail(3)

<details>
<summary>💬 Discuss</summary>

- Monday's Census API answered with a list of lists. What shape is FRED's answer, and which is easier to turn into a DataFrame?
- The Census Bureau's estimate for Boulder County in 2020–2024 was $103,994; FRED's for 2024 is $103,293. The chapter says they come from different programs. Which would you cite, and how would you say which it is?
- Why fix `observation_end`, when the series only goes to 2024 anyway?

</details>

## Step 9 — Let the server compute: `units` (8 minutes)

We'll ask FRED for observations several more times this week, each time with one parameter changed. So gather Step 8's lines into one function, `fred_observations()`, the chapter's name: whatever changes from call to call is an argument, and `**params` passes along any other parameters. Then ask for the same series twice more, with [`units`](https://fred.stlouisfed.org/docs/api/fred/series_observations.html#units) set to `"chg"`, the change in dollars from the year before, and to `"pch"`, the percent change, as the chapter's ["Let the Server Compute: `units`"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#let-the-server-compute-units) does. One series, three answers, and FRED computed two of them.

In [ ]:
# Define fred_observations(series_id, **params): Step 8's observation lines, gathered
#   Add series_id, api_key, and file_type to params; request; build and convert the DataFrame; return it
def fred_observations(series_id, **params):
    params.update(series_id=series_id, api_key=fred_key, file_type="json")
    data = requests.get(f"{fred_base}/series/observations", params=params, headers=HEADERS).json()
    df = pd.DataFrame(data["observations"])[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

In [ ]:
# change: fred_observations() with units="chg"; growth: with units="pch"; both with observation_end="2024-12-31"
# views: income's date's year and value, change's value, and growth's value as four columns; show the last 3 rows
change = fred_observations("MHICO08013A052NCEN", units="chg", observation_end="2024-12-31")
growth = fred_observations("MHICO08013A052NCEN", units="pch", observation_end="2024-12-31")
views = pd.DataFrame({"year": income["date"].dt.year, "dollars": income["value"],
                      "change": change["value"], "percent change": growth["value"]})
views.tail(3).round(1)

### ⏱ Microsprint (4 minutes, alone or in pairs)

Check what the server computed against the dollars.

**Do:** find the years in which Boulder County's median household income *fell*: the rows of `views` whose `change` is below zero.
**Share:** how many years, which ones, and the largest fall in dollars.

In [ ]:
# Microsprint: falls, the rows of views whose "change" is below zero; show them
falls = views[views["change"] < 0]
falls.round(1)

<details>
<summary>💬 Discuss</summary>

- Did the falls cluster around any events you know? What would you check before saying they were caused by them?
- FRED computed `change` from the same dollars we have. When is it worth asking the server for something you could compute yourself?
- `fred_observations()` has no error handling: if FRED answered with a maintenance page, which line would fail, and what would its error say?

</details>

## End of Part 2

On Friday, Part 3 asks FRED to change a series' frequency and compares Denver with the nation; then it asks the EIA to filter its data for us. It starts with its own setup step, which defines `fred_observations()` again, so you don't need today's cells to have run.

# Part 3 — Friday: more of FRED; then the EIA, and let the server filter

FRED computes one more thing for us: a lower frequency, and inflation from it. Then the EIA (U.S. Energy Information Administration) cuts its enormous tables down to the rows and columns we name before it sends them.

## Step 10 — Set up again (3 minutes)

Part 3 starts fresh. Import what we need, define the User-Agent, check and read `FRED_KEY` and `EIA_KEY` (Step 0 says what to do if one is missing), and run the given cell, Wednesday's `fred_observations()` from Step 9 and the chapter's ["Let the Server Compute: `units`"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#let-the-server-compute-units).

In [ ]:
# Import os, requests, pandas (as pd), and matplotlib.pyplot (as plt); define HEADERS as in Step 0, with your email
# For "FRED_KEY" and "EIA_KEY", print whether each is set or MISSING; then read them into fred_key and eia_key
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
for name in ["FRED_KEY", "EIA_KEY"]:
    print(name + ":", "set" if name in os.environ else "MISSING")
fred_key = os.environ["FRED_KEY"]
eia_key = os.environ["EIA_KEY"]

In [ ]:
# Only for a key that's MISSING, if you can't restart Jupyter now: ask for it with getpass
from getpass import getpass
fred_key = getpass("Paste your FRED key: ")
eia_key = getpass("Paste your EIA key: ")

In [ ]:
# Given: Wednesday's function from Step 9, which asks FRED for a series' observations as a DataFrame
fred_base = "https://api.stlouisfed.org/fred"

def fred_observations(series_id, **params):
    params.update(series_id=series_id, api_key=fred_key, file_type="json")
    data = requests.get(f"{fred_base}/series/observations", params=params, headers=HEADERS).json()
    df = pd.DataFrame(data["observations"])[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

<details>
<summary>💬 Discuss</summary>

- FRED and the EIA both call their key parameter `api_key`. What would you need to change to use `fred_observations()` with another API?

</details>

## Step 11 — Let the server compute: `frequency` and `aggregation_method` (10 minutes)

A series' frequency is how often it is measured, and FRED can lower it before it sends the data, as the chapter's ["Let the Server Compute: `frequency` and `aggregation_method`"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#let-the-server-compute-frequency-and-aggregation_method) shows. Take one series through three requests: `CPIAUCNS`, the Consumer Price Index (CPI) for all urban consumers in the United States, which the Bureau of Labor Statistics (BLS) publishes every month. Ask for it as published; then with [`frequency="a"` and `aggregation_method="avg"`](https://fred.stlouisfed.org/docs/api/fred/series_observations.html#frequency), which turn each year's twelve months into their mean; then with `units="pc1"` as well, the percent change of each year's mean from the year before's, which is the year's inflation. FRED applies the frequency first, then the units.

In [ ]:
# years: observation_start="2015-01-01", observation_end="2025-12-31"; monthly: CPIAUCNS as published
# annual: frequency="a", aggregation_method="avg"; inflation: the same, plus units="pc1"
# table: annual's year, annual's value, and inflation's value; show it
years = {"observation_start": "2015-01-01", "observation_end": "2025-12-31"}
monthly = fred_observations("CPIAUCNS", **years)
annual = fred_observations("CPIAUCNS", frequency="a", aggregation_method="avg", **years)
inflation = fred_observations("CPIAUCNS", frequency="a", aggregation_method="avg", units="pc1", **years)
table = pd.DataFrame({"year": annual["date"].dt.year, "mean index": annual["value"],
                      "inflation": inflation["value"]})
table.round(1)

### ⏱ Microsprint (4 minutes, alone or in pairs)

Check the server's math, as the chapter does, on the same series.

**Do:** from `monthly`, compute each year's mean with [`groupby()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) on the year, then each year's percent change with [`pct_change()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.pct_change.html) times 100.
**Share:** your numbers for 2024 and 2025 beside FRED's, and why your 2015 is missing.

In [ ]:
# Microsprint: by_hand: monthly's values grouped by the date's year, averaged
# check: by_hand and its pct_change() * 100 as two columns; show the last 3 rows
by_hand = monthly.groupby(monthly["date"].dt.year)["value"].mean()
check = pd.DataFrame({"mean index": by_hand, "inflation": by_hand.pct_change() * 100})
check.tail(3).round(1)

<details>
<summary>💬 Discuss</summary>

- Did your numbers match FRED's? If they hadn't, which would you trust, and how would you find out?
- Why can FRED give 2015's inflation when our pandas version can't?
- `aggregation_method` could also be `sum` or `eop`, the end of the period. Which makes sense for a price index, and which for something counted, like jobs added?

</details>

## Step 12 — Denver against the nation (5 minutes)

With one series understood, comparing two is asking each the same question. The BLS (Bureau of Labor Statistics) also publishes a CPI (Consumer Price Index) for the Denver area, which FRED carries as `CUUSA433SA0`. It is annual already, so it needs only `units="pc1"`. Merge it with Step 11's `inflation` on the date, and draw the chapter's paired bars: one pair a year, with a dashed line at 2 percent, the Federal Reserve's target, as in the chapter's [figure 11.6](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#fig-cpi-denver-national). A DataFrame with one row per year and one column per place is what [`DataFrame.plot.bar()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.bar.html) draws as pairs.

In [ ]:
# denver: CUUSA433SA0 with units="pc1" and **years; cpi: denver merged with inflation on "date", suffixes ("_denver", "_national")
# pairs: cpi indexed by its date's year, the two value columns, renamed "Denver-Aurora-Lakewood" and "United States"
denver = fred_observations("CUUSA433SA0", units="pc1", **years)
cpi = denver.merge(inflation, on="date", suffixes=("_denver", "_national"))
pairs = cpi.set_index(cpi["date"].dt.year)[["value_denver", "value_national"]]
pairs.columns = ["Denver-Aurora-Lakewood", "United States"]

In [ ]:
# Plot pairs as bars (rot=0); a dashed gray line at 2 labeled "2% target"; a title, a y label, no x label, and a legend
ax = pairs.plot.bar(figsize=(8, 4), rot=0, width=0.8)
ax.axhline(y=2, color="gray", linestyle="--", linewidth=1, label="2% target")
ax.set(title="Annual inflation (CPI-U, percent change from the year before)", ylabel="Percent", xlabel="")
ax.legend()
plt.show()

<details>
<summary>💬 Discuss</summary>

- In which years did Denver's prices rise faster than the nation's, and in which slower?
- Denver's series is annual and the nation's monthly. What would have gone wrong if we had merged the monthly series with the annual one, without `frequency="a"`?
- Housing is the largest part of the CPI. What would you ask FRED next to test whether rent explains the gap?

</details>

## Step 13 — EIA: routes that describe themselves (6 minutes)

The EIA's (U.S. Energy Information Administration's) API is a tree of **routes**, paths such as `electricity/retail-sales/` ([documentation](https://www.eia.gov/opendata/documentation.php)). Ask for a route without asking for its data, and it describes itself: at the top, the routes under it; further down, what its data can be filtered by and what it holds, as the chapter's ["Routes That Describe Themselves"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#routes-that-describe-themselves) shows. Every answer is inside the JSON's `"response"`. It's Monday's idea, *what can I ask for?*, from a different API.

In [ ]:
# eia_base = "https://api.eia.gov/v2"; request f"{eia_base}/electricity/" with params={"api_key": eia_key} and HEADERS
# Print each of the JSON's ["response"]["routes"]: its "id" and its "name"
eia_base = "https://api.eia.gov/v2"
electricity = requests.get(f"{eia_base}/electricity/", params={"api_key": eia_key}, headers=HEADERS).json()["response"]
for child in electricity["routes"]:
    print(child["id"], "|", child["name"])

In [ ]:
# meta: the "response" of f"{eia_base}/electricity/retail-sales/", requested the same way
# Print its "frequency" ids and its "facets" ids; then each column of its "data", with that column's "units"
meta = requests.get(f"{eia_base}/electricity/retail-sales/", params={"api_key": eia_key}, headers=HEADERS).json()["response"]
print("Frequencies:", [f["id"] for f in meta["frequency"]])
print("Facets:", [f["id"] for f in meta["facets"]])
for column, about in meta["data"].items():
    print(column, "|", about["units"])

<details>
<summary>💬 Discuss</summary>

- On Monday, the Census Bureau's catalogs were separate files, `groups.json` and each group's own. Here the route describes itself. Which would you rather explore, and why?
- The chapter's tour shows the EIA's API Dashboard. Where do its menus come from?
- What does a **facet** do that a data column doesn't?

</details>

## Step 14 — Let the server filter (8 minutes)

Add `data/` to the route, and it returns rows; each parameter narrows them, as the chapter's ["Let the Server Filter"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#let-the-server-filter) shows. `data[]` names the columns, `facets[stateid][]` and `facets[sectorid][]` keep only the values you list, `start` and `end` fix the years, and `sort` orders the rows. The square brackets are part of the parameters' names ([documentation](https://www.eia.gov/opendata/documentation.php)), and `requests` encodes them for you. Ask for the average price of electricity in Colorado's homes, every year from 2001 to 2025.

In [ ]:
# params: frequency="annual", data[]="price", facets[stateid][]="CO", facets[sectorid][]="RES", start="2001", end="2025"
#   and sort[0][column]="period", sort[0][direction]="asc"; with api_key=eia_key
# Request f"{eia_base}/electricity/retail-sales/data/"; prices: its "response"; print its "total"; co_price: a DataFrame of its "data"
params = {"frequency": "annual", "data[]": "price", "facets[stateid][]": "CO", "facets[sectorid][]": "RES",
          "start": "2001", "end": "2025", "sort[0][column]": "period", "sort[0][direction]": "asc",
          "api_key": eia_key}
prices = requests.get(f"{eia_base}/electricity/retail-sales/data/", params=params, headers=HEADERS).json()["response"]
print("Rows:", prices["total"])
co_price = pd.DataFrame(prices["data"])
co_price[["period", "stateid", "sectorName", "price", "price-units"]].tail(3)

### ⏱ Microsprint (4 minutes, alone or in pairs)

Change one facet.

**Do:** ask for the same price in another sector (`COM`, commercial, or `IND`, industrial), or for homes in another state (its two-letter code, such as `UT` or `NM`). The key asks for Colorado's commercial price.
**Share:** your sector or state, its 2025 price, and how it compares with Colorado's homes.

In [ ]:
# Microsprint: set params["facets[sectorid][]"] (or params["facets[stateid][]"]) to your choice; request again
# Show the last 3 rows of period, stateid, sectorName, and price
params["facets[sectorid][]"] = "COM"
other = requests.get(f"{eia_base}/electricity/retail-sales/data/", params=params, headers=HEADERS).json()["response"]
pd.DataFrame(other["data"])[["period", "stateid", "sectorName", "price"]].tail(3)

<details>
<summary>💬 Discuss</summary>

- How many rows came back, out of a route that holds every state and sector by month, quarter, and year? What did filtering on the server save?
- The prices arrive as strings. Which line would fail if you tried to average them?
- The EIA sends at most 5,000 rows a request. The chapter's ["Paging with `offset` and `length`"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#paging-with-offset-and-length) asks for Colorado's monthly generation, 9,289 rows. Before writing a loop for that, what would you try first?

</details>

## Step 15 — What did we learn? (3 minutes)

In a few sentences, as comments:

- Step 3 asked the API for the website's number. Did they match, and what would you check first if they hadn't?
- The week's three APIs each carried an idea: what can I ask for, let the server compute, let the server filter. Which one saved you the most code, and where?
- In Step 7, how many of Colorado's places had no estimate, and what would their mean income have looked like with the codes left in?
- Which of this week's numbers would be different if you ran this notebook next week, and which wouldn't? Why?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday):

- Step 1, load your keys without printing them: Step 0's check, `"set" if ... in os.environ else "MISSING"`, and its fixes for a `KeyError`.
- Step 2, an ACS variable for every Colorado county: Steps 1 and 2 to find the group and the variable, Step 3's `county:013` with `in="state:08"`, and Step 7's wildcard, which asked for every place.
- Step 3, build the DataFrame, convert, sort, and print the extremes: Step 3's `pd.to_numeric`, and Step 7's `describe()` and `mask()` for the codes that aren't numbers.
- Step 4, a bar chart of the top 20 counties: Step 5's bar chart.
- Step 5, a related FRED series for Colorado over time: Steps 8 and 9's `fred_observations()`; find the series on FRED's website, as the chapter's tour does.
- Step 6, interpret: the Discuss questions after Steps 3, 6, 7, and 11.

In [ ]:
# Your answers here, as comments
# ANSWERS

## Friday: from the lab to the textbook (15 minutes)

The last 15 minutes go to the textbook. Look back at what the lab turned up in [chapter 11](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html):

- **A cell that broke.** On Monday, `os.environ["CENSUS_KEY"]` stopped with a `KeyError` for some of us after `export`. Does the chapter's "Setting Environment Variables" explain why, and what to do? Where would you add a sentence?
- **A sentence that confused.** Which step needed the website beside the API to make sense? Does the chapter's tour do that for the Census table, and could it do more?
- **An output that has changed.** The EIA's 2025 figures are preliminary. Compare Step 14's prices with the chapter's.
- **A microsprint that would make a good exercise.** Searching `groups.json` for your own topic (Step 1), or comparing two cities' margins (Step 6), could be an Additional Exercise.

The [revision framework](https://github.com/cuinfoscience/INFO4617-Fall2026/blob/main/handouts/common/revision-framework.md) says how to propose a change, and what each kind needs to be accepted.